## เริ่ม Model

## ใช้ Dataset ใน Sklearn มีหลายแบบโหลดไม่เหมือนกัน

## กรณีงานจริง กำหนด Feature และ Target

In [ ]:
# กำหนด features และ target
# X = df[["column1", "column2", "column3", "column4"]]  # features
# y = df["ผลลัพธ์"]    # target
# กำหนด target_names เอง
# target_names = ["cat", "dog", "rabbit"]

## เริ่ม dataset ตัวอย่าง

In [23]:
## Forest Covertypes dataset โหลดและแปลงกลับเป็น column ต้นทางที่ยังไม่ได้ one hot encoder
## Target classes (7 ชนิดของป่า)   581012 rows × 14 columns
## covertype คือ target
# Spruce/Fir ป่าสนสปรูซและเฟอร์ 
# Lodgepole Pine ป่าสน Lodgepole 
# Ponderosa Pine ป่าสน Ponderosa 
# Cottonwood/Willow ป่าต้น Cottonwood และ Willow 
# Aspen ป่า Aspen 
# Douglas-fir ป่าสน Douglas-f 
# Krummholz ป่าพุ่มไม้เตี้ยที่ขึ้นในพื้นที่สูงมาก 


In [ ]:
import pandas as pd
forest = pd.read_csv(r'D:\Forest Covertypes.csv')
forest = forest.set_index('Index')

# forest.info()
forest.head(5)
# forest.index
# forest.columns
# forest.shape


## สำรวจข้อมูลก่อนเข้า Model เพิ่มส่วน EDA อีกครั้ง

In [ ]:
# จัดการค่า Missing
# จัดการค่า Outlier
# จัดการค่า Dupplicate

forest.info()
# forest.isnull().sum()


## แยก Train กับ Test ที่ขั้นตอนนี้เลย

In [ ]:
## แบ่งข้อมูลเป็น 3 ชุด 1.Train set  2.Validation set  3.Test set
## เราจะแบ่งมาตรฐาน 60 Train  Validation 10  Test 30

from sklearn.model_selection import train_test_split

# แยก features และ target
X = forest.drop(columns=["Cover_Type"])   # ลบ target ออก
y = forest["Cover_Type"]                  # target

# แยก Train+ Validation /Test 70:30  stratify= ตัวแปร target y หรือ 'covertype'  รักษาสัดส่วน class ให้ข้อมูลแต่ละ Class เท่าต้นฉบับ
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.3, random_state=42,stratify=y)

# รอบสอง: แบ่ง train (60%) และ validation (10%) จาก X_temp (70%)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1429, random_state=42, stratify=y_temp)


# X_train
# X_val
# y_train
# y_val


## ต่อด้วย One hot Encoding หรือ Label Encoding

In [ ]:
## จัดการ column กลุ่ม Category  ด้วย One hot Encoding pd.get_dummies(trainset,columns = catgorycols)
## ทำทั้ง Train set และ Test set
## ใช้ One hot Encoding เป็นมาตรฐาน ถึงจะเป็น label ที่มีลำดับหรือไม่มีลำดับก็ใช้ได้ดี
## ใช้ Label Encoding กับ Label ที่มีลำดับเท่านั้น
## text columns ที่เป็น free text เช่น ชื่อ-นามสกุล, หมู่บ้าน, อาคาร → ตัดออก
## ส่วนใหญ่ไม่มีความหมายเชิง category ที่ช่วยโมเดล และจำนวน unique values มักเยอะมากจนเกินเหตุตัดออก
## one-hot ไม่ practical → ตัดออกไปได้เลย

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# สมมติว่ามี categorical columns
cat_cols = ["Wilderness_Area", "Soil_Type"]

# Train และ Test set ใช้ OneHotEncoder จาก Pandas  pd.get_dummies()
X_train_final = pd.get_dummies(X_train, columns=cat_cols)
X_val_final = pd.get_dummies(X_val, columns=cat_cols)
X_test_final  = pd.get_dummies(X_test, columns=cat_cols)


X_trainTreebase = X_train_final
X_valTreebase = X_val_final
X_testTreebase = X_test_final

# target y ทั้ง Train และ Test encode ตามตัวเลข ผลลัพธ์เป็น array โยนเข้า Dataframe ตั้งชื่อ Cover_type ตามเดิม
le = LabelEncoder()
ytrainall = pd.DataFrame(le.fit_transform(y_train),columns =["Cover_Type"])
yvalall = pd.DataFrame(le.fit_transform(y_val),columns =["Cover_Type"])
ytestall =  pd.DataFrame(le.transform(y_test),columns =["Cover_Type"])
ytrainall

## แบบใช้ cat.codes เปลี่ยนเป็นตัวเลขเหมือนกันไม่ได้ใช้ label encoder
# ytrainall = pd.DataFrame(y_train.astype('category').cat.codes,columns =["Cover_Type"])
# ytestall = pd.DataFrame(y_test.astype('category').cat.codes,columns=["Cover_Type"])

# Feature Scaling

In [ ]:
# จัดการกลุ่ม Column ตัวเลข ที่อาจมีฐานข้อมูล หน่วย ไม่เท่ากัน
# ****เราทำ Feature Scaling เพื่อปรับหน่วยตัวเลขแต่ละ Column ให้ใกล้เคียงกัน****
#    ****กรณีหน่วยมันเป็นตัวเดียวกันอยู่แล้วไม่ต้องทำตีความยาก****

# standard normailize  เหมาะกับ Logistic Regression(Linear Classification) , SVM (Distance-based ใช้ระยะทาง)
# Min-Max normailize   เหมาะกับ Neural Network, KNN (Distance-based ใช้ระยะทาง Euclidean)
# RobustScaler normailize  เหมาะกับ Dataset ที่มี outlier เยอะๆ
# Treebase Model ไม่ต้อง feature Scaling Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost
## *****Tree-based models (Random Forest, XGBoost) **** → ไม่จำเป็นต้องทำ scaling เพราะไม่อ่อนไหวต่อสเกล


In [9]:
# 10 column ที่เป็นตัวเลข Float
numeric_cols = ["Elevation", "Aspect", "Slope",
                "Horizontal_Distance_To_Hydrology",
                "Vertical_Distance_To_Hydrology",
                "Horizontal_Distance_To_Roadways",
                "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
                "Horizontal_Distance_To_Fire_Points"]

# แยก X_numtrain 10 column ตัวเลข X_catgorytrain ที่เหลือของ x ที่เป็น category
X_numtrain = X_train_final[numeric_cols]
X_catgorytrain = X_train_final.drop(columns = numeric_cols)

# แยก X_numtest 10 column ตัวเลข X_catgorytest ที่เหลือของ x ที่เป็น category
X_numtest = X_test_final[numeric_cols]
X_catgorytest = X_test_final.drop(columns = numeric_cols)

# แยก X_val 10 column ตัวเลข X_catgorytest ที่เหลือของ x ที่เป็น category
X_numval = X_val_final[numeric_cols]
X_catgoryval = X_val_final.drop(columns = numeric_cols)


In [10]:
## แบบที่ 1  StandardScaler  ใช้กับ Logistic Regression, SVM
# เอา X_num_scaled ที่แปลงแล้ว เข้า dataframe และรวมกับ X_catgory ที่ไม่รวม Cover_Type
#  index=X_num.index เป็นการบอกว่า Scaler ที่แปลงแล้วให้ใช้ index เดิม

# บน Train set  .fit_transform
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
scaler1 = StandardScaler()            # สร้างครั้งเดียวพอ
X_num_trainscaled1 = scaler1.fit_transform(X_numtrain)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtrain และรวมกับ X_catgorytrain
X_num_trainscaled_df1 = pd.DataFrame(X_num_trainscaled1, columns=numeric_cols,index=X_numtrain.index)
xtrainscaler1 = pd.concat([X_num_trainscaled_df1, X_catgorytrain], axis=1)


# บน Test set  .transform
X_num_testscaled1 = scaler1.transform(X_numtest)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtest และรวมกับ X_catgorytest
X_num_testscaled_df1 = pd.DataFrame(X_num_testscaled1, columns=numeric_cols,index=X_numtest.index)
xtestscaler1 = pd.concat([X_num_testscaled_df1, X_catgorytest], axis=1)

# บน Val set  .transform
X_num_valscaled1 = scaler1.transform(X_numval)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtest และรวมกับ X_catgorytest
X_num_valscaled_df1 = pd.DataFrame(X_num_valscaled1, columns=numeric_cols,index=X_numval.index)
xvalscaler1 = pd.concat([X_num_valscaled_df1, X_catgoryval], axis=1)

## ได้ที่จะใช้
# xtrainscaler1
# xtestscaler1
# xvalscaler1

In [11]:
## แบบที่ 2 Neural Network, K-Nearest Neightbor
## แบบที่ 2 บน Train set  .fit_transform

from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
scaler2 = MinMaxScaler()

# บน Train set  .fit_transform
X_num_trainscaled2 = scaler2.fit_transform(X_numtrain)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtrain และรวมกับ X_catgorytrain
X_num_trainscaled_df2 = pd.DataFrame(X_num_trainscaled2, columns=numeric_cols,index=X_numtrain.index)
xtrainscaler2 = pd.concat([X_num_trainscaled_df2, X_catgorytrain], axis=1)

## บน Test set  .transform
X_num_testscaled2 = scaler2.transform(X_numtest)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtest และรวมกับ X_catgorytest
X_num_testscaled_df2 = pd.DataFrame(X_num_testscaled2, columns=numeric_cols,index=X_numtest.index)
xtestscaler2 = pd.concat([X_num_testscaled_df2, X_catgorytest], axis=1)

# บน Val set  .transform
X_num_valscaled2 = scaler2.transform(X_numval)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtest และรวมกับ X_catgorytest
X_num_valscaled_df2 = pd.DataFrame(X_num_valscaled2, columns=numeric_cols,index=X_numval.index)
xvalscaler2 = pd.concat([X_num_valscaled_df2, X_catgoryval], axis=1)


# ได้ผลลัพธ์
# xtrainscaler2
# xtestscaler2
# xvalscaler2

In [12]:
## แบบที่ 3 # RobustScaler  Outlier มาก

scaler3 = RobustScaler()
X_num_trainscaled3 = scaler3.fit_transform(X_numtrain)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtrain และรวมกับ X_catgorytrain
X_num_trainscaled_df3 = pd.DataFrame(X_num_trainscaled3, columns=numeric_cols,index=X_numtrain.index)
xtrainscaler3 = pd.concat([X_num_trainscaled_df3, X_catgorytrain], axis=1)


## บน Test set  .transform
X_num_testscaled3 = scaler3.transform(X_numtest)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtest และรวมกับ X_catgorytest
X_num_testscaled_df3 = pd.DataFrame(X_num_testscaled3, columns=numeric_cols,index=X_numtest.index)
xtestscaler3 = pd.concat([X_num_testscaled_df3, X_catgorytest], axis=1)

# บน Val set  .transform
X_num_valscaled3 = scaler3.transform(X_numval)

# หลัง Scaler จับเข้า Dataframe column ตามที่แบ่งด้านบน index ตาม X_numtest และรวมกับ X_catgorytest
X_num_valscaled_df3 = pd.DataFrame(X_num_valscaled3, columns=numeric_cols,index=X_numval.index)
xvalscaler3 = pd.concat([X_num_valscaled_df3, X_catgoryval], axis=1)

# ได้ผลลัพธ์
# xtrainscaler3
# xtestscaler3
# xvalscaler3

## Feature Engineering

In [ ]:
# สร้างฟีเจอร์ใหม่จากข้อมูลเดิม							
	วันเกิด → อายุ							
	ทำ binning เช่น อายุแบ่งเป็นช่วง (0–18, 19–35, 36–60, 60+) 							
	เป็น category กลุ่มอายุแต่ละ Gen							
	วันที่ซื้อสินค้า → วันในสัปดาห์, เดือน, ฤดูกาล			จันทร์ อังคาร พุธ พฤหัส ศุกร์ เสาร์ อาทิตย์				เดือน
	ที่อยู่ → รหัสไปรษณีย์, ภูมิภาค							
#  แปลงข้อมูลให้อยู่ในรูปที่โมเดลใช้ได้							
	ข้อความ → TF-IDF, Word Embedding			ใน NLP อันนี้ไม่ต้อง				
	หมวดหมู่ → One-hot encoding หรือ Label encoding							
#   รวมฟีเจอร์เพื่อสร้างข้อมูลเชิงลึก							
	รายได้ต่อเดือน ÷ จำนวนสมาชิกครอบครัว → รายได้ต่อหัว							
	ยอดขาย ÷ จำนวนวัน → ยอดขายเฉลี่ยต่อวัน							
#   ลด noise							
	ตัดค่า Outlier ที่ผิดปกติจริงๆออก มันเกิด noise							


## Feature Selection

In [ ]:
## ใช้วิธี Correlation ดูเบื้องต้น แต่สรุปไม่ได้ว่า feature ไหนสำคัญกับ model

# สมมติว่า X_train เป็น DataFrame และ y_train เป็น Series ของ target
# รวม X_train และ y_train เข้าด้วยกันเพื่อคำนวณ correlation

dfcor = pd.concat([X_trainTreebase,ytrainall], axis=1)
dfcor.tail()
corr_matrix = dfcor.corr()
print(corr_matrix)  

In [ ]:
## ใช้วิธี Embedding โยนเข้า Xgboost  เพื่อหา feature importance
## กลุ่ม Treebase model ใช้ x train y train ที่ไม่ปรับ scale

# รอบแรก: train เพื่อดู feature importance
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=200, random_state=42)

model.fit(X_trainTreebase,ytrainall)

# ดู feature importance
# model.feature_importances_ คืนค่ามาเป็น array index column 0 1 2 3 4 ที่สำคัญ
# f for f เก็บค่าที่ผ่านเงื่อนไขเข้า list
# zip(feature_names, importances) ชื่อ columns กับ ลำดับ ได้ค่า imp แต่ละ column
importances = model.feature_importances_
feature_names = X_trainTreebase.columns
important_features = [f for f, imp in zip(feature_names, importances) if imp > 0.01]

print(pd.DataFrame(important_features))



## Hyperparameter และ Cross Validation

In [ ]:
## StratifiedKFold จะพยายามรักษาสัดส่วน class ให้ใกล้เคียงกันในแต่ละ fold 
# (สำคัญมากในงาน classification ที่ class imbalance)
# shuffle=True → สุ่มข้อมูลก่อนแบ่ง fold
# random_state=42 → ทำให้การสุ่ม reproducible (ได้ผลเหมือนเดิมทุกครั้ง)


from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# กำหนด CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# สร้างโมเดล TreeBased
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": lgb.LGBMClassifier(random_state=42)
}

# วนลูปทำ cross_val_score
for name, model in models.items():
    scores = cross_val_score(model, X_trainTreebase, ytrainall, cv=cv, scoring='accuracy')
    print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}")


Decision Tree: mean=0.9223, std=0.0008


c:\Users\Nat44\py311_env\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Nat44\py311_env\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Nat44\py311_env\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Nat44\py311_env\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# สร้างโมเดล กลุ่ม Distance1
modelsD1 = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(random_state=42),
}

# วนลูปทำ cross_val_score
for name, model in modelsD1.items():
    scores = cross_val_score(model,xtrainscaler1, ytrainall, cv=cv, scoring='accuracy')
    print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}")

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

# สร้างโมเดล กลุ่ม Distance2
modelsD2 = {
            "KNN": KNeighborsClassifier()
}

# วนลูปทำ cross_val_score
for name, model in modelsD2.items():
    scores = cross_val_score(model, xtrainscaler2, ytrainall, cv=cv, scoring='accuracy')
    print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}")

#   Train Model จริง

In [43]:
# รอบสอง: train จริงด้วยเฉพาะ important features

# ตัวแปรกลุ่ม Treebase ไม่ Scaler
X_trainTree = X_trainTreebase[important_features]
X_testTree = X_testTreebase[important_features]
ytrainall
ytestall 

# ตัวแปรกลุ่ม Distance Scaler
# แบบที่ 1  StandardScaler  ใช้กับ Logistic Regression, SVM
X_trainDis1 = xtrainscaler1[important_features]
X_testDis1 = xtrainscaler1[important_features]

# แบบที่ 2 Neural Network, K-Nearest Neightbor
X_trainDis2 = xtrainscaler2[important_features]
X_testDis2 = xtrainscaler2[important_features]

## แบบที่ 3 # RobustScaler  Outlier มาก
X_trainDis3 = xtrainscaler3[important_features]
X_testDis3 = xtrainscaler3[important_features]


In [44]:
## กลุ่ม Tree base model  
# 1.XGBoost
from xgboost import XGBClassifier
modelxgb = XGBClassifier(n_estimators=500, random_state=42)
modelxgb.fit(X_trainTree,ytrainall)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
# 2.Decision Tree
from sklearn.tree import DecisionTreeClassifier
modeldt = DecisionTreeClassifier(random_state=42)
modeldt.fit(X_trainTree, ytrainall)

In [ ]:
# 3.Random Forrest
from sklearn.ensemble import RandomForestClassifier
modelrf = RandomForestClassifier(n_estimators=500, random_state=42)
modelrf.fit(X_trainTree, ytrainall)

In [ ]:
# 4.LightGbm
from lightgbm import LGBMClassifier
modellgb = LGBMClassifier(n_estimators=500, random_state=42)
modellgb.fit(X_trainTree, ytrainall)

In [ ]:
#  กลุ่ม Distance Model และ Linear Classification
# 5.K-Nerest Neighbors  ใช้ Data ที่ปรับ MinMaxScaler()
from sklearn.neighbors import KNeighborsClassifier
modelknn = KNeighborsClassifier(n_neighbors=5)  # ค่า k ปรับได้
modelknn.fit(X_trainDis2, ytrainall)


In [ ]:
# 6.Support Vector Machine
from sklearn.svm import SVC
modelsvm = SVC(kernel='rbf', random_state=42)  # kernel ปรับได้ เช่น 'linear'
modelsvm.fit(X_trainDis1, ytrainall)


In [ ]:
# 7. Logistic Regression
from sklearn.linear_model import LogisticRegression
modellog = LogisticRegression(random_state=42, max_iter=1000)
modellog.fit(X_trainDis1, ytrainall)

In [ ]:
# Predict 7 Model
# y_predxgb = modelxgb.predict(X_testTree)

# y_preddt = modeldt.predict(X_testTree)

# y_predrf = modelrf.predict(X_testTree)

# y_predlgb = modellgb.predict(X_testTree)

# y_predknn = modelknn.predict(X_testDis2)

# y_predsvm = modelsvm.predict(X_testDis1)

# y_predlog = modellog.predict(X_testDis1)


In [46]:
from sklearn.metrics import classification_report
print(classification_report(ytestall, y_predxgb))

              precision    recall  f1-score   support

           0       0.90      0.81      0.85      2848
           1       0.85      0.81      0.83       824
           2       0.89      0.85      0.87      5210
           3       0.97      0.95      0.96      6153
           4       0.92      0.95      0.94     84991
           5       0.92      0.93      0.93     10726
           6       0.93      0.91      0.92     63552

    accuracy                           0.93    174304
   macro avg       0.91      0.89      0.90    174304
weighted avg       0.93      0.93      0.93    174304



In [ ]:
# print(classification_report(ytestall, y_preddt))
# print(classification_report(ytestall, y_predrf))
# print(classification_report(ytestall, y_predlgb))
# print(classification_report(ytestall, y_predknn))
# print(classification_report(ytestall, y_predsvm))
# print(classification_report(ytestall, y_predlog))

              precision    recall  f1-score   support

           0       0.92      0.83      0.87      2848
           1       0.88      0.84      0.86       824
           2       0.91      0.86      0.88      5210
           3       0.97      0.95      0.96      6153
           4       0.96      0.97      0.96     84991
           5       0.93      0.95      0.94     10726
           6       0.96      0.95      0.96     63552

    accuracy                           0.96    174304
   macro avg       0.93      0.91      0.92    174304
weighted avg       0.96      0.96      0.96    174304

